In [79]:
import pandas as pd
import numpy as np

In [80]:
def load_signal_csv(path, col_name):
    df = pd.read_csv(path, header=None)
    start_time = float(df.iloc[0, 0])
    freq = float(df.iloc[1, 0])
    values = df.iloc[2:, 0].astype(float).values
    timestamps = start_time + np.arange(len(values)) / freq 

    df = pd.DataFrame({ 
        "time": timestamps,
        col_name: values
    })
    df["time"] = pd.to_datetime(df["time"], unit="s", utc=True)
    df["time"] = df["time"].dt.tz_convert("America/Chicago")

    exam_date = df["time"].dt.date.iloc[0]

    exam_start = pd.Timestamp(
        year=exam_date.year,
        month=exam_date.month,
        day=exam_date.day,
        hour=9,
        minute=0,
        second=0,
        tz="America/Chicago"
    )

    df = df[df["time"] >= exam_start]
    return df

In [81]:

def load_acc_csv(path):
    df = pd.read_csv(path, header=None)

    start_time = float(df.iloc[0, 0])
    freq = float(df.iloc[1, 0])

    values = df.iloc[2:].astype(float).values  

    timestamps = start_time + np.arange(len(values)) / freq

    df_acc = pd.DataFrame(values, columns=['x', 'y', 'z'])
    df_acc['time'] = timestamps

    df_acc["time"] = pd.to_datetime(df_acc["time"], unit="s", utc=True)
    df_acc["time"] = df_acc["time"].dt.tz_convert("America/Chicago")

    exam_date = df_acc["time"].dt.date.iloc[0]

    exam_start = pd.Timestamp(
        year=exam_date.year,
        month=exam_date.month,
        day=exam_date.day,
        hour=9,
        minute=0,
        second=0,
        tz="America/Chicago"
    )

    df_acc = df_acc[df_acc["time"] >= exam_start]

    df_acc['acc_mag'] = np.sqrt(
        df_acc['x']**2 + df_acc['y']**2 + df_acc['z']**2
    )

    return df_acc[['time', 'acc_mag']]

In [82]:
def resample_1hz(df):
    df['time'] = pd.to_datetime(df['time'], unit='s')
    df = df.set_index('time')

    df_resampled = df.resample('1s').mean()

    return df_resampled.reset_index()

In [83]:
import os
students = [f"S{i}" for i in range(1, 11)]
exams = ["Midterm 1", "Midterm 2", "Final"]
base_path = "Data"

In [84]:
marks_dict = {
    ("S1", "Midterm 1"): 78,
    ("S2", "Midterm 1"): 82,
    ("S3", "Midterm 1"): 77,
    ("S4", "Midterm 1"): 75,
    ("S5", "Midterm 1"): 67,
    ("S6", "Midterm 1"): 71,
    ("S7", "Midterm 1"): 64,
    ("S8", "Midterm 1"): 92,
    ("S9", "Midterm 1"): 80,
    ("S10", "Midterm 1"): 89,

    ("S1", "Midterm 2"): 82,
    ("S2", "Midterm 2"): 85,
    ("S3", "Midterm 2"): 90,
    ("S4", "Midterm 2"): 77,
    ("S5", "Midterm 2"): 77,
    ("S6", "Midterm 2"): 64,
    ("S7", "Midterm 2"): 33,
    ("S8", "Midterm 2"): 88,
    ("S9", "Midterm 2"): 39,
    ("S10", "Midterm 2"): 64,

    ("S1", "Final"): 182/200*100,
    ("S2", "Final"): 180/200*100,
    ("S3", "Final"): 188/200*100,
    ("S4", "Final"): 149/200*100,
    ("S5", "Final"): 157/200*100,
    ("S6", "Final"): 175/200*100,
    ("S7", "Final"): 110/200*100,
    ("S8", "Final"): 184/200*100,
    ("S9", "Final"): 126/200*100,
    ("S10", "Final"): 116/200*100,
}

In [85]:
all_data = []

for student in students:
    for exam in exams:
        path = os.path.join(base_path, student, exam)

        eda = resample_1hz(load_signal_csv(os.path.join(path, "EDA.csv"), "eda"))
        bvp = resample_1hz(load_signal_csv(os.path.join(path, "BVP.csv"), "bvp"))
        temp = resample_1hz(load_signal_csv(os.path.join(path, "TEMP.csv"), "temp"))
        hr = resample_1hz(load_signal_csv(os.path.join(path, "HR.csv"), "hr"))
        acc = resample_1hz(load_acc_csv(os.path.join(path, "ACC.csv")))
        df = eda.merge(bvp,on='time',how='outer')
        df = df.merge(temp,on='time',how='outer')
        df = df.merge(hr,on='time',how='outer')
        df = df.merge(acc,on='time',how='outer')
        df = df.dropna(how='any')

        df = df.sort_values("time")
        df["student"] = student
        df["exam"] = exam

        all_data.append(df)

In [86]:
finaldf= pd.concat(all_data)
finaldf

,time,eda,bvp,temp,hr,acc_mag,student,exam
0,2018-10-13 09:00:00-05:00,0.840230,0.006875,27.13,83.72,64.771668,S1,Midterm 1
1,2018-10-13 09:00:01-05:00,0.836386,0.016562,27.09,83.48,64.770225,S1,Midterm 1
2,2018-10-13 09:00:02-05:00,0.907184,0.007969,27.11,83.27,64.753476,S1,Midterm 1
3,2018-10-13 09:00:03-05:00,0.940821,-0.054375,27.13,83.02,64.752259,S1,Midterm 1
4,2018-10-13 09:00:04-05:00,0.910067,-0.026250,27.11,82.78,64.690983,S1,Midterm 1
...,...,...,...,...,...,...,...,...
23067,2018-12-05 16:53:21-06:00,0.020820,-6.644844,22.53,156.95,62.903943,S10,Final
23068,2018-12-05 16:53:22-06:00,0.020820,0.461719,22.51,158.10,63.300591,S10,Final
23069,2018-12-05 16:53:23-06:00,0.021140,-3.839531,22.53,159.28,63.414938,S10,Final
23070,2018-12-05 16:53:24-06:00,0.020500,3.465000,22.51,160.32,67.761859,S10,Final


In [87]:
# finaldf.groupby()

In [88]:
finaldf["marks"] = finaldf.apply(
    lambda row: marks_dict[(row["student"], row["exam"])],
    axis=1
)


In [90]:
finaldf.size

3416022

In [91]:
mid1_df = finaldf[finaldf["exam"] == "Midterm 1"]
mid2_df = finaldf[finaldf["exam"] == "Midterm 2"]
final_only_df = finaldf[finaldf["exam"] == "Final"]

In [92]:
def convert_to_window(df):
    midtermwindows=[]
    windowsize=5
    groups = df.groupby(["student", "exam"])
    print(list(groups.groups.keys()))

    for i in groups.groups.keys():
        grp=groups.get_group(i)
        grp=grp.sort_values("time").reset_index(drop=True)
        
        for j in range(0,len(grp) - windowsize + 1,windowsize):
            window=grp.iloc[j:j+windowsize]
            features={
                "student":window["student"].iloc[0],
                "exam":window["exam"].iloc[0],
                "marks":window["marks"].iloc[0],
                
                "eda": window["eda"].mean(),

                "hr": window["hr"].mean(),

                "temp": window["temp"].mean(),
                "bvp": window["bvp"].mean(),
                "acc": window["acc_mag"].mean(),

            }
            midtermwindows.append(features)

    midterm_windows_df = pd.DataFrame(midtermwindows)
    return midterm_windows_df

In [93]:
mid1_window=convert_to_window(mid1_df)
mid2_window=convert_to_window(mid2_df)
final_window=convert_to_window(final_only_df)

[('S1', 'Midterm 1'), ('S10', 'Midterm 1'), ('S2', 'Midterm 1'), ('S3', 'Midterm 1'), ('S4', 'Midterm 1'), ('S5', 'Midterm 1'), ('S6', 'Midterm 1'), ('S7', 'Midterm 1'), ('S8', 'Midterm 1'), ('S9', 'Midterm 1')]
[('S1', 'Midterm 2'), ('S10', 'Midterm 2'), ('S2', 'Midterm 2'), ('S3', 'Midterm 2'), ('S4', 'Midterm 2'), ('S5', 'Midterm 2'), ('S6', 'Midterm 2'), ('S7', 'Midterm 2'), ('S8', 'Midterm 2'), ('S9', 'Midterm 2')]
[('S1', 'Final'), ('S10', 'Final'), ('S2', 'Final'), ('S3', 'Final'), ('S4', 'Final'), ('S5', 'Final'), ('S6', 'Final'), ('S7', 'Final'), ('S8', 'Final'), ('S9', 'Final')]


In [94]:
final_df = pd.concat([mid1_window, mid2_window, final_window])

In [95]:
final_df.size

607200

In [96]:
groups=final_df.groupby(["student","exam"])

In [146]:
allgroups=[]
for (a,b),i in groups:
    grp=groups.get_group((a,b))
    dataframe={
        ("acc","std"):(grp["acc"].mean(),grp["acc"].std()),
        ("eda","std"):(grp["eda"].mean(),grp["eda"].std()),
        ("bvp","std"):(grp["bvp"].mean(),grp["bvp"].std()),
        ("temp","std"):(grp["temp"].mean(),grp["temp"].std()),
        ("hr","std"):(grp["hr"].mean(),grp["hr"].std()),
    }
    allpoints=[]
    for key,value in dataframe.items():
        threshold=4*value[1] + value[0]
        allpoints.append(grp[grp[key[0]]>=threshold])
    data=pd.concat(allpoints)
    data=data.drop_duplicates()
    finaldataframe={
        "acc":data["acc"].mean(),
        "eda":data["eda"].mean(),
        "bvp":data["bvp"].mean(),
        "temp":data["temp"].mean(),
        "hr":data["hr"].mean(),
        "class":1 if marks_dict[(a,b)]>75 else 0,
        "examination":b
    }
    # print(finaldataframe)
    # df_final=pd.DataFrame(finaldataframe,i)
    allgroups.append(finaldataframe)
real_df=pd.DataFrame(allgroups)

In [147]:
real_df["examination"] = real_df["examination"].map({"Midterm 1": 0, "Midterm 2": 1, "Final":2})


In [148]:
real_df

,acc,eda,bvp,temp,hr,class,examination
0,69.785333,0.093899,8.377772,26.689962,104.863500,1,2
1,71.599880,0.352815,0.984200,27.881778,93.554667,1,0
2,70.411492,0.137739,5.912268,25.006457,104.493943,1,1
3,66.169900,0.521285,12.917206,28.319667,89.197667,0,2
4,66.261816,0.313497,1.646353,27.766485,85.420667,1,0
5,71.086510,0.148193,8.370908,25.811379,95.071517,0,1
6,65.593436,0.406006,5.425136,32.849121,103.248113,1,2
7,65.085033,0.579465,1.778757,30.709756,102.982195,1,0
8,64.295529,0.422001,6.773811,29.698476,119.593476,1,1
9,70.723527,0.573652,16.632439,28.438483,89.930207,1,2


In [149]:
real_df

,acc,eda,bvp,temp,hr,class,examination
0,69.785333,0.093899,8.377772,26.689962,104.863500,1,2
1,71.599880,0.352815,0.984200,27.881778,93.554667,1,0
2,70.411492,0.137739,5.912268,25.006457,104.493943,1,1
3,66.169900,0.521285,12.917206,28.319667,89.197667,0,2
4,66.261816,0.313497,1.646353,27.766485,85.420667,1,0
5,71.086510,0.148193,8.370908,25.811379,95.071517,0,1
6,65.593436,0.406006,5.425136,32.849121,103.248113,1,2
7,65.085033,0.579465,1.778757,30.709756,102.982195,1,0
8,64.295529,0.422001,6.773811,29.698476,119.593476,1,1
9,70.723527,0.573652,16.632439,28.438483,89.930207,1,2


In [150]:
X=real_df.drop(columns=["class"])
y=real_df["class"]

In [151]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [161]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

In [162]:

model = LogisticRegression()
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [164]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

y_pred = model.predict(X_test)

print("acc:", accuracy_score(y_test, y_pred))
print("precision:", precision_score(y_test, y_pred))
print("recall:", recall_score(y_test, y_pred))


acc: 0.8333333333333334
precision: 0.8
recall: 1.0
